# 📦 Brazilian E-Commerce: Delivery Duration, Delay Risk & Satisfaction Prediction
### End-to-End Machine Learning System Using the Olist Dataset
**Author:** Salman  
**Dataset:** [Olist Brazilian E-Commerce Dataset (Kaggle)](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)  
**Tech Stack:** Python, Kagglehub, Scikit-Learn, Pandas, NumPy, Matplotlib, Seaborn, FastAPI, Streamlit

---
### Executive Summary
In e-commerce, accurate logistics estimation and proactive delay management directly impact customer loyalty and operational profitability. This project builds a multi-task predictive machine learning engine on over 96,000 real-world Brazilian e-commerce orders to solve three core problems:
1. **Delivery Duration (Days)**: Predicting actual transit duration from purchase to customer delivery.
2. **Late Delivery Risk**: Predicting the probability that an order arrives past the estimated SLA deadline.
3. **Customer Satisfaction (Review Score 1-5)**: Predicting customer sentiment based on delivery speed, freight share, and product specifications.

In [ ]:
# Core Imports
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('All libraries imported successfully!')

## 1. Dataset Download via Kagglehub
We use `kagglehub.dataset_download` to programmatically retrieve the latest version of the Brazilian E-Commerce dataset.

In [ ]:
# Download latest version of Olist dataset
dataset_path = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print('Path to dataset files:', dataset_path)
dataset_dir = Path(dataset_path)

# List downloaded CSV files
csv_files = list(dataset_dir.glob('*.csv'))
for f in csv_files:
    print(f' - {f.name} ({f.stat().st_size / (1024*1024):.2f} MB)')

## 2. Data Ingestion & Relational Merging
The Olist dataset consists of 8 relational tables linked by primary and foreign keys (`order_id`, `customer_id`, `product_id`, `seller_id`). We merge these tables into an integrated modeling dataset.

In [ ]:
# Load raw CSV tables
orders = pd.read_csv(dataset_dir / 'olist_orders_dataset.csv')
order_items = pd.read_csv(dataset_dir / 'olist_order_items_dataset.csv')
products = pd.read_csv(dataset_dir / 'olist_products_dataset.csv')
customers = pd.read_csv(dataset_dir / 'olist_customers_dataset.csv')
sellers = pd.read_csv(dataset_dir / 'olist_sellers_dataset.csv')
reviews = pd.read_csv(dataset_dir / 'olist_order_reviews_dataset.csv')
payments = pd.read_csv(dataset_dir / 'olist_order_payments_dataset.csv')
category_trans = pd.read_csv(dataset_dir / 'product_category_name_translation.csv')

print(f'Orders: {orders.shape}')
print(f'Items: {order_items.shape}')
print(f'Products: {products.shape}')

## 3. Data Cleaning & Feature Engineering
We engineer domain-specific logistics features:
- **Delivery Duration (`delivery_days`)**: Continuous target in days.
- **Late Delivery (`is_late`)**: Binary classification target ($1$ if actual delivery > estimated delivery date, else $0$).
- **Intra-State Delivery (`is_same_state`)**: Binary indicator whether customer and seller reside in the same Brazilian state.
- **Freight Burden (`freight_ratio`)**: Proportion of total order spend dedicated to freight charges.
- **Product Volume & Weight**: Physical packaging constraints.
- **Temporal Dynamics**: Purchase hour, day of week, month, and weekend indicator.

In [ ]:
# 1. Filter delivered orders with valid timestamps
df_orders = orders[orders['order_status'] == 'delivered'].copy()
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_orders['order_delivered_customer_date'] = pd.to_datetime(df_orders['order_delivered_customer_date'])
df_orders['order_estimated_delivery_date'] = pd.to_datetime(df_orders['order_estimated_delivery_date'])
df_orders = df_orders.dropna(subset=['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date'])

# Target Variables
df_orders['delivery_days'] = (df_orders['order_delivered_customer_date'] - df_orders['order_purchase_timestamp']).dt.total_seconds() / (24 * 3600)
df_orders = df_orders[(df_orders['delivery_days'] > 0) & (df_orders['delivery_days'] <= 100)].copy()
df_orders['is_late'] = (df_orders['order_delivered_customer_date'] > df_orders['order_estimated_delivery_date']).astype(int)
df_orders['estimated_delivery_days'] = (df_orders['order_estimated_delivery_date'] - df_orders['order_purchase_timestamp']).dt.total_seconds() / (24 * 3600)

# Temporal features
df_orders['purchase_hour'] = df_orders['order_purchase_timestamp'].dt.hour
df_orders['purchase_dayofweek'] = df_orders['order_purchase_timestamp'].dt.dayofweek
df_orders['purchase_month'] = df_orders['order_purchase_timestamp'].dt.month
df_orders['is_weekend'] = df_orders['purchase_dayofweek'].isin([5, 6]).astype(int)

# 2. Process Products & Order Items
products = products.merge(category_trans, on='product_category_name', how='left')
products['product_category'] = products['product_category_name_english'].fillna('other')
products['product_volume_cm3'] = (products['product_length_cm'].fillna(20) * products['product_height_cm'].fillna(15) * products['product_width_cm'].fillna(15))
products['product_weight_g'] = products['product_weight_g'].fillna(products['product_weight_g'].median())

items_merged = order_items.merge(products[['product_id', 'product_category', 'product_weight_g', 'product_volume_cm3']], on='product_id', how='left')
items_merged = items_merged.merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='left')

items_agg = items_merged.groupby('order_id').agg({
    'price': 'sum',
    'freight_value': 'sum',
    'order_item_id': 'count',
    'product_weight_g': 'mean',
    'product_volume_cm3': 'mean',
    'product_category': 'first',
    'seller_state': 'first'
}).reset_index().rename(columns={'price': 'total_price', 'freight_value': 'total_freight', 'order_item_id': 'item_count'})
items_agg['freight_ratio'] = items_agg['total_freight'] / (items_agg['total_price'] + items_agg['total_freight'] + 1e-5)

# 3. Process Payments & Reviews
payments_agg = payments.groupby('order_id').agg({
    'payment_installments': 'max',
    'payment_type': lambda x: x.mode()[0] if not x.empty else 'credit_card'
}).reset_index()

reviews_agg = reviews.groupby('order_id').agg({'review_score': 'mean'}).reset_index()

# 4. Final Merge
df = df_orders.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='inner')
df = df.merge(items_agg, on='order_id', how='inner')
df = df.merge(payments_agg, on='order_id', how='left')
df = df.merge(reviews_agg, on='order_id', how='left')

df['seller_state'] = df['seller_state'].fillna('SP')
df['customer_state'] = df['customer_state'].fillna('SP')
df['is_same_state'] = (df['customer_state'] == df['seller_state']).astype(int)
df['payment_installments'] = df['payment_installments'].fillna(1).clip(1, 24)
df['payment_type'] = df['payment_type'].fillna('credit_card')
df['review_score'] = df['review_score'].fillna(df['review_score'].median())

print(f'Clean Master Modeling Dataset Shape: {df.shape}')
df.head()

## 4. Exploratory Data Analysis (EDA)
Let us inspect the distribution of delivery duration, on-time delivery rates, and the relationship between shipping delays and customer review scores.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Delivery Duration Distribution
sns.histplot(df['delivery_days'], bins=40, kde=True, color='#2563eb', ax=axes[0])
axes[0].set_title('Delivery Duration Distribution (Days)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Days from Purchase to Delivery')

# 2. On-Time vs Late Deliveries
late_counts = df['is_late'].value_counts(normalize=True) * 100
axes[1].pie(late_counts, labels=['On-Time', 'Late'], autopct='%1.1f%%', colors=['#10b981', '#ef4444'], startangle=90, explode=[0, 0.1])
axes[1].set_title('On-Time Delivery Rate (%)', fontsize=12, fontweight='bold')

# 3. Review Score by Delivery Status
sns.barplot(x='is_late', y='review_score', data=df, palette=['#10b981', '#ef4444'], ax=axes[2])
axes[2].set_title('Customer Review Score vs Delay Status', fontsize=12, fontweight='bold')
axes[2].set_xticklabels(['On-Time Deliveries', 'Late Deliveries'])
axes[2].set_ylabel('Mean Review Score (1-5)')

plt.tight_layout()
plt.show()

## 5. Model 1: Delivery Duration Regressor
Predicting continuous delivery time in days using `HistGradientBoostingRegressor` with column transformers.

In [ ]:
num_features = [
    'total_price', 'total_freight', 'freight_ratio', 'item_count',
    'product_weight_g', 'product_volume_cm3', 'payment_installments',
    'purchase_hour', 'purchase_dayofweek', 'purchase_month',
    'is_weekend', 'is_same_state', 'estimated_delivery_days'
]
cat_features = ['product_category', 'customer_state', 'seller_state', 'payment_type']
all_features = num_features + cat_features

X = df[all_features]
y_duration = df['delivery_days']
y_late = df['is_late']
y_review = df['review_score']

X_train, X_test, y_dur_train, y_dur_test, y_late_train, y_late_test, y_rev_train, y_rev_test = train_test_split(
    X, y_duration, y_late, y_review, test_size=0.2, random_state=42, stratify=y_late
)

# Preprocessing pipeline
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='other')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_features)
])

# Regressor Pipeline
dur_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(max_iter=150, learning_rate=0.08, max_leaf_nodes=31, random_state=42))
])

print('Training Delivery Duration Regressor...')
dur_pipeline.fit(X_train, y_dur_train)
y_dur_pred = dur_pipeline.predict(X_test)

mae = mean_absolute_error(y_dur_test, y_dur_pred)
rmse = np.sqrt(mean_squared_error(y_dur_test, y_dur_pred))
r2 = r2_score(y_dur_test, y_dur_pred)
print(f'Delivery Duration Model -> MAE: {mae:.2f} days | RMSE: {rmse:.2f} days | R2: {r2:.3f}')

## 6. Model 2: Late Delivery Risk Classifier
Predicting the probability that an order exceeds its promised SLA date using `HistGradientBoostingClassifier` with balanced class weights.

In [ ]:
late_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08, class_weight='balanced', random_state=42))
])

print('Training Late Delivery Classifier...')
late_pipeline.fit(X_train, y_late_train)
y_late_pred = late_pipeline.predict(X_test)
y_late_proba = late_pipeline.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_late_test, y_late_pred)
auc = roc_auc_score(y_late_test, y_late_proba)
f1 = f1_score(y_late_test, y_late_pred)
print(f'Late Delivery Classifier -> ROC-AUC: {auc:.3f} | Accuracy: {acc:.3f} | F1: {f1:.3f}')
print('\nClassification Report:\n', classification_report(y_late_test, y_late_pred))

## 7. Model 3: Customer Satisfaction (Review Score) Regressor
Predicting review score (1–5) based on transit speed, delay gap, and product characteristics.

In [ ]:
# Add predicted duration and delay gap as features for sentiment
X_train_rev = X_train.copy()
X_test_rev = X_test.copy()
X_train_rev['pred_delivery_days'] = dur_pipeline.predict(X_train)
X_test_rev['pred_delivery_days'] = y_dur_pred
X_train_rev['pred_delay_gap'] = X_train_rev['pred_delivery_days'] - X_train_rev['estimated_delivery_days']
X_test_rev['pred_delay_gap'] = X_test_rev['pred_delivery_days'] - X_test_rev['estimated_delivery_days']

rev_num_features = num_features + ['pred_delivery_days', 'pred_delay_gap']
rev_preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), rev_num_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='other')), ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_features)
])

rev_pipeline = Pipeline([
    ('preprocessor', rev_preprocessor),
    ('regressor', HistGradientBoostingRegressor(max_iter=120, learning_rate=0.08, random_state=42))
])

print('Training Customer Satisfaction Regressor...')
rev_pipeline.fit(X_train_rev, y_rev_train)
y_rev_pred = rev_pipeline.predict(X_test_rev)

rev_mae = mean_absolute_error(y_rev_test, y_rev_pred)
rev_rmse = np.sqrt(mean_squared_error(y_rev_test, y_rev_pred))
print(f'Customer Satisfaction Regressor -> MAE: {rev_mae:.2f} stars | RMSE: {rev_rmse:.2f}')

## 8. Inference Function & Interactive Demonstration
Let us test live predictions on sample real-world order scenarios.

In [ ]:
def predict_ecommerce_order(order_dict):
    """Run inference across all 3 trained models for an incoming order."""
    is_same_state = 1 if order_dict['customer_state'].upper() == order_dict['seller_state'].upper() else 0
    freight_ratio = order_dict['total_freight'] / (order_dict['total_price'] + order_dict['total_freight'] + 1e-5)
    
    input_row = pd.DataFrame([{
        'total_price': order_dict['total_price'],
        'total_freight': order_dict['total_freight'],
        'freight_ratio': freight_ratio,
        'item_count': order_dict.get('item_count', 1),
        'product_weight_g': order_dict['product_weight_g'],
        'product_volume_cm3': order_dict['product_volume_cm3'],
        'payment_installments': order_dict.get('payment_installments', 1),
        'purchase_hour': order_dict.get('purchase_hour', 14),
        'purchase_dayofweek': order_dict.get('purchase_dayofweek', 2),
        'purchase_month': order_dict.get('purchase_month', 6),
        'is_weekend': 0,
        'is_same_state': is_same_state,
        'estimated_delivery_days': order_dict['estimated_delivery_days'],
        'product_category': order_dict['product_category'],
        'customer_state': order_dict['customer_state'].upper(),
        'seller_state': order_dict['seller_state'].upper(),
        'payment_type': order_dict.get('payment_type', 'credit_card')
    }])
    
    pred_days = float(dur_pipeline.predict(input_row)[0])
    late_proba = float(late_pipeline.predict_proba(input_row)[0][1]) * 100
    
    input_rev = input_row.copy()
    input_rev['pred_delivery_days'] = pred_days
    input_rev['pred_delay_gap'] = pred_days - order_dict['estimated_delivery_days']
    pred_review = float(rev_pipeline.predict(input_rev)[0])
    
    return {
        'predicted_delivery_days': round(pred_days, 1),
        'estimated_sla_days': order_dict['estimated_delivery_days'],
        'late_risk_pct': round(late_proba, 1),
        'late_risk_level': 'LOW RISK' if late_proba < 20 else ('MODERATE RISK' if late_proba < 50 else 'HIGH RISK'),
        'predicted_review_score': round(max(1.0, min(5.0, pred_review)), 1)
    }

# Sample Demonstration
sample_order = {
    'product_category': 'bed_bath_table',
    'total_price': 149.90,
    'total_freight': 18.20,
    'product_weight_g': 1200.0,
    'product_volume_cm3': 6500.0,
    'customer_state': 'SP',
    'seller_state': 'SP',
    'estimated_delivery_days': 15.0
}

print('Demonstration Inference Result:')
result = predict_ecommerce_order(sample_order)
for k, v in result.items():
    print(f'  {k}: {v}')